# Group by, fetch and export

Summarize sales by city, filter groups with HAVING, choose Python result formats and export CSV or JSON from the same in-memory database.

[Watch lesson 6](https://asqueel.org/learn/05-relations/)

**Run the cells from top to bottom in a fresh Python kernel.** Each notebook includes its own starting state. SQLite runs in memory; restarting the kernel discards the database. To repeat the exercise, restart the kernel and run all cells. Colab execution requires a Google account.

© 2026 Softwell S.r.l. · Apache-2.0

## Before the examples: `_ = ...` is optional

It assigns a return value we do not need, preventing its automatic display in the Python console or notebook. The call still runs normally. This is a Python convention, not an Asqueel requirement. When you need the result, use a meaningful variable name.

## Prepare the notebook

The video starts in a terminal. Here, install the same official package in the notebook kernel. No virtual environment or separate Python console is needed in Colab.

In [ ]:
%pip install -q "asqueel[migration]==0.6.0"

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

## Recreate the starting point

The previous lessons are rebuilt below. The four fictional CSV files are included in this notebook and written into a new temporary folder. No downloads or existing database are needed.

In [ ]:
#@title Prepare the sample CSV files
from pathlib import Path
import os
import tempfile
_lesson_folder = tempfile.TemporaryDirectory(prefix="asqueel-notebook-")
os.chdir(_lesson_folder.name)
_csv_files = {'customers.csv': 'id,name,city\n1,Ada,Florence\n2,Grace,London\n3,Linus,Helsinki\n4,Guido,Amsterdam\n5,Elena,Florence\n6,Alan,Manchester\n7,Margaret,New York\n8,Barbara,Boston\n', 'invoice_rows.csv': 'id,invoice_id,product_code,quantity,unit_price,total\n1,101,TEA,2,3.50,7.00\n2,101,COF,1,6.90,6.90\n3,101,MUG,4,5.25,21.00\n4,102,BTEA,2,3.20,6.40\n5,102,HTEA,3,4.10,12.30\n6,102,HON,4,7.50,30.00\n7,103,SUG,3,2.30,6.90\n8,103,CHO,4,3.90,15.60\n9,103,BIS,1,2.80,2.80\n10,104,JAM,4,4.60,18.40\n11,104,OAT,1,2.40,2.40\n12,104,JUI,2,2.70,5.40\n13,105,WAT,1,0.90,0.90\n14,105,LEM,2,1.80,3.60\n15,105,KET,3,32.90,98.70\n16,106,POT,2,18.50,37.00\n17,106,PLT,3,6.40,19.20\n18,106,BWL,4,7.80,31.20\n19,107,SPO,3,9.50,28.50\n20,107,TRY,4,14.90,59.60\n21,107,JAR,1,8.20,8.20\n22,108,NAP,4,12.00,48.00\n23,108,TWL,1,5.60,5.60\n24,108,FLT,2,2.10,4.20\n25,109,TIN,1,6.70,6.70\n26,109,MAT,2,11.40,22.80\n27,109,CAF,3,29.90,89.70\n28,110,BOT,2,13.50,27.00\n29,110,STR,3,4.80,14.40\n30,110,SCL,4,24.90,99.60\n31,111,TEA,3,3.50,10.50\n32,111,COF,4,6.90,27.60\n33,111,MUG,1,5.25,5.25\n34,112,BTEA,4,3.20,12.80\n35,112,HTEA,1,4.10,4.10\n36,112,HON,2,7.50,15.00\n', 'invoices.csv': 'id,number,customer_id,invoice_date\n101,INV-001,1,2026-09-28\n102,INV-002,2,2026-09-29\n103,INV-003,3,2026-09-30\n104,INV-004,4,2026-10-01\n105,INV-005,5,2026-10-02\n106,INV-006,6,2026-10-03\n107,INV-007,7,2026-10-04\n108,INV-008,8,2026-10-05\n109,INV-009,1,2026-10-06\n110,INV-010,2,2026-10-07\n111,INV-011,5,2026-10-08\n112,INV-012,3,2026-10-09\n', 'products.csv': 'code,name,category,stock\nBTEA,Black tea,Drinks,30\nHTEA,Herbal tea,Drinks,18\nHON,Honey,Food,22\nSUG,Cane sugar,Food,40\nCHO,Dark chocolate,Food,35\nBIS,Biscuits,Food,28\nJAM,Strawberry jam,Food,16\nOAT,Oat drink,Drinks,32\nJUI,Apple juice,Drinks,26\nWAT,Mineral water,Drinks,60\nLEM,Lemonade,Drinks,30\nKET,Kettle,Kitchen,9\nPOT,Teapot,Kitchen,14\nPLT,Plate,Kitchen,36\nBWL,Bowl,Kitchen,24\nSPO,Spoon set,Kitchen,20\nTRY,Serving tray,Kitchen,11\nJAR,Storage jar,Kitchen,25\nNAP,Cotton napkins,Kitchen,18\nTWL,Tea towel,Kitchen,30\nFLT,Coffee filters,Kitchen,45\nTIN,Tea tin,Kitchen,15\nMAT,Table mats,Kitchen,16\nCAF,Coffee maker,Kitchen,7\nBOT,Water bottle,Kitchen,21\nSTR,Tea strainer,Kitchen,23\nSCL,Kitchen scale,Kitchen,8\n'}
for _name, _text in _csv_files.items():
    _ = Path(_name).write_text(_text, encoding="utf-8")


In [ ]:
#@title Recreate the previous lessons

######## Import our two tools ########

from asqueel import AsqueelDb, SqlDatabaseConfig


######## Choose SQLite in memory ########

model = SqlDatabaseConfig()
root = model.source.db(implementation="sqlite")
_ = root.connection(name=":memory:", implementation="sqlite")


######## Describe a product table ########

shop = root.packages().package("shop", sqlschema="shop")
product = shop.table("product", pkey="code")
_ = product.column("code", size=":10")
_ = product.column("name", size=":60")


######## Create that table ########

db = AsqueelDb(model)
print(db.migrate().commands)


######## Add our first product ########

products = db.table("shop.product")
products.insert({"code": "TEA", "name": "Tea"})
db.commit()


######## Read our product ########

products.query(columns="$code, $name").fetch()


######## Change its name ########

products.update({"name": "Green tea"}, pkey="TEA")
db.commit()


######## Check the update ########

products.query(columns="$code, $name").fetch()
db.commit()


######## Track category and stock ########

_ = product.column("category", size=":30")
_ = product.column("stock", dtype="I")


######## Load the updated model ########

db.reloadModel(model)


######## Add those database columns ########

print(db.migrate().commands)


######## Check our original row ########

products = db.table("shop.product")
products.query(columns="$name, $stock").fetch()
db.commit()


######## Give tea its stock ########

products.update({"category": "Drinks", "stock": 8}, pkey="TEA")
db.commit()


######## Read the new values ########

products.query(columns="$name, $category, $stock").fetch()
db.commit()


######## Add products from CSV ########

from io import StringIO
data = StringIO(
    "code,name,category,stock\n"
    "COF,Coffee,Drinks,24\n"
    "MUG,Mug,Kitchen,12\n"
)


######## Import those rows ########

products.sql_insert(data, format="csv")
db.commit()


######## List just the drinks ########

products.query(
    columns="$name, $stock",
    where="$category = :category", category="Drinks",
    order_by="$stock DESC",
).fetch()
db.commit()

# Continue with lesson two.

######## Check our starting point ########

products.query(columns="$code, $name").fetch()
db.commit()


######## Describe our customers ########

customer = shop.table("customer", pkey="id")
_ = customer.column("id", dtype="I")
_ = customer.column("name", size=":60")
_ = customer.column("city", size=":60")


######## Create the customer table ########

db.reloadModel(model)
print(db.migrate().commands)


######## Describe a dated invoice ########

invoice = shop.table("invoice", pkey="id")
_ = invoice.column("id", dtype="I")
_ = invoice.column("number", size=":20")
_ = invoice.column("invoice_date", dtype="D")


######## Connect invoices to customers ########

customer_id = invoice.column("customer_id", dtype="I")
_ = customer_id.relation(
    "shop.customer.id", foreign_key=True, relation_name="invoices",
)


######## Describe an invoice row ########

invoice_row = shop.table("invoice_row", pkey="id")
_ = invoice_row.column("id", dtype="I")
_ = invoice_row.column("quantity", dtype="I")
_ = invoice_row.column("unit_price", dtype="N", size="12,2")
_ = invoice_row.column("total", dtype="N", size="12,2")


######## Connect the row at both ends ########

invoice_id = invoice_row.column("invoice_id", dtype="I")
_ = invoice_id.relation(
    "shop.invoice.id", foreign_key=True, relation_name="rows",
)
product_code = invoice_row.column("product_code", size=":10")
_ = product_code.relation(
    "shop.product.code", foreign_key=True, relation_name="rows",
)


######## Apply the new model ########

db.reloadModel(model)
print(db.migrate().commands)


######## Get fresh table handles ########

products = db.table("shop.product")
customers = db.table("shop.customer")
invoices = db.table("shop.invoice")
rows = db.table("shop.invoice_row")


######## Inspect the row file ########

from pathlib import Path
print("\n".join(Path("invoice_rows.csv").read_text().splitlines()[:4]))


######## Expand the product catalogue ########

with open("products.csv", newline="", encoding="utf-8") as source:
    products.sql_insert(source)

db.commit()


######## Count the catalogue ########

products.query().count()
db.commit()


######## Import customers and their cities ########

with open("customers.csv", newline="", encoding="utf-8") as source:
    customers.sql_insert(source)

db.commit()
customers.query(columns="$name, $city", limit=3).fetch()
db.commit()


######## Import the invoices ########

with open("invoices.csv", newline="", encoding="utf-8") as source:
    invoices.sql_insert(source)

db.commit()


######## Import the invoice rows ########

with open("invoice_rows.csv", newline="", encoding="utf-8") as source:
    rows.sql_insert(source)

db.commit()


######## Check how much we imported ########

invoices.query().count()
rows.query().count()
db.commit()


######## Read invoices with their customers ########

invoices.query(
    columns="$number, $invoice_date, "
            "@customer_id.name, @customer_id.city",
    limit=3,
).fetch()
db.commit()


######## Inspect the first invoice ########

rows.query(
    columns="@product_code.name, "
            "$quantity, $unit_price, $total",
    where="$invoice_id = :invoice", invoice=101,
).fetch()
db.commit()


######## Read across both connections ########

rows.query(
    columns="@invoice_id.number, "
            "@invoice_id.@customer_id.city, "
            "@product_code.name, $total",
    limit=3,
).fetch()
db.commit()

# Continue with lesson three.

######## Find invoices for Florence ########

invoices.query(
    columns="$number",
    where="@customer_id.city = :city", city="Florence",
).fetch()


######## Sort invoices by city ########

invoices.query(
    columns="$number, @customer_id.city",
    order_by="@customer_id.city", limit=4,
).fetch()


######## Show the newest rows first ########

rows.query(
    columns="@invoice_id.invoice_date, $total",
    order_by="@invoice_id.invoice_date DESC", limit=3,
).fetch()


######## Find rows for Florence ########

rows.query(
    columns="@product_code.name, $quantity",
    where="@invoice_id.@customer_id.city = :city",
    city="Florence", limit=3,
).fetch()


######## Combine the conditions ########

from datetime import date
rows.query(
    columns="@product_code.name, $total",
    where="@invoice_id.@customer_id.city = :city "
          "AND @invoice_id.invoice_date >= :since",
    city="Florence", since=date(2026, 10, 1),
    order_by="@invoice_id.invoice_date DESC", limit=4,
).fetch()


######## Close this read transaction ########

db.commit()

# Continue with lesson 4.

######## Call this path city ########

_ = invoice_row.aliasColumn(
    "city", relation_path="@invoice_id.@customer_id.city",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")


######## Read and filter with that name ########

rows.query(
    columns="$city, @product_code.name",
    where="$city = :city", city="Florence", limit=3,
).fetch()
db.commit()


######## Calculate quantity times price ########

_ = invoice_row.formulaColumn(
    "calculated_total",
    sql_formula="ROUND($quantity * $unit_price, 2)",
    dtype="N",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")


######## Read the calculated amounts ########

rows.query(
    columns="$quantity, $unit_price, $calculated_total",
    where="$invoice_id = :invoice", invoice=101,
    order_by="$id",
).fetch()
db.commit()


######## Check whether the database needs changing ########

db.migration_plan().empty


######## Temporarily change the quantity to three ########

rows.update({"quantity": 3}, pkey=1)
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=1,
).fetch()


######## Undo that change ########

db.rollback()

# Continue with lesson 5.

######## Give each invoice a calculated total ########

_ = invoice.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice_row",
        columns="ROUND(SUM($calculated_total), 2)",
        where="$invoice_id = #THIS.id",
    ),
)
db.reloadModel(model)
invoices = db.table("shop.invoice")


######## Read those totals as ordinary fields ########

invoices.query(
    columns="$number, $total", order_by="$id", limit=3,
).fetch()
db.commit()


######## Total the invoices of each customer ########

_ = customer.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice",
        columns="ROUND(SUM($total), 2)",
        where="$customer_id = #THIS.id",
    ),
)
db.reloadModel(model)
customers = db.table("shop.customer")


######## Read each customer's sales total ########

customers.query(
    columns="$name, $total", order_by="$id", limit=3,
).fetch()
db.commit()


######## Total the sales of each product ########

_ = product.formulaColumn(
    "total", dtype="N",
    select=dict(
        table="shop.invoice_row",
        columns="ROUND(SUM($calculated_total), 2)",
        where="$product_code = #THIS.code",
    ),
)
db.reloadModel(model)
products = db.table("shop.product")
rows = db.table("shop.invoice_row")


######## Read the product totals ########

products.query(
    columns="$code, $name, $total",
    where="$code IN :codes", codes=["TEA", "COF", "MUG"],
    order_by="$code",
).fetch()
db.commit()


######## Summarize each customer's invoices ########

customers = db.table("shop.customer")
customers.query(
    columns="$name, @invoices.aggr("
            "sum($total), count(), avg($total))",
    order_by="$id", limit=3,
).fetch()
db.commit()


######## Count the rows of each invoice too ########

invoices = db.table("shop.invoice")
invoices.query(
    columns="$number, $total, @rows.aggr(count())",
    order_by="$id", limit=3,
).fetch()
db.commit()


######## Average the recorded selling prices ########

products.query(
    columns="$code, @rows.aggr(count(), avg($unit_price))",
    where="$code IN :codes", codes=["TEA", "COF", "MUG"],
    order_by="$code",
).fetch()
db.commit()


######## Name the average price so we can filter it ########

_ = product.formulaColumn(
    "average_price", dtype="N",
    select=dict(
        table="shop.invoice_row", columns="AVG($unit_price)",
        where="$product_code = #THIS.code",
    ),
)
db.reloadModel(model)
products = db.table("shop.product")
rows = db.table("shop.invoice_row")


######## Filter by average price ########

products.query(
    columns="$code, $name, $average_price",
    where="$average_price BETWEEN :min_price AND :max_price",
    min_price=3.2, max_price=4.8,
    order_by="$average_price, $code",
).fetch()
db.commit()


## 1 — What we'll learn in this lesson

Welcome back! Here's what we'll learn. Group sales by city, filter those groups, choose how Python receives the results, and export a report. We keep the same database in memory.

```text
What we'll learn in this lesson
GROUP BY → one summary per city
WHERE and HAVING → before and after grouping
fetch → rows, dictionaries, groups and JSON
Export → CSV and JSON files

Same SQLite database in memory
```

## 2 — Group invoice rows by customer city

Let's total sales by city. City is the alias from lesson four. Group by puts rows from the same city together. Sum adds their calculated amounts. We get one summary per city, not one result per invoice row.

In [ ]:
######## Total sales by city ########

rows = db.table("shop.invoice_row")
products = db.table("shop.product")
rows.query(
    columns="$city, SUM($calculated_total)",
    group_by="$city", order_by="$city", limit=3,
).fetch()

Let's name the report columns. We just ran without aliases. Here, optional AS labels make the report easier to read and export: sales and row count. Count measures invoice rows, not customers or units sold. The database still holds all thirty-six original rows.

In [ ]:
######## Name the report columns ########

report = rows.query(
    columns="$city, ROUND(SUM($calculated_total), 2) AS sales, "
            "COUNT(*) AS row_count",
    group_by="$city", order_by="$city",
)
report.fetch()

## 3 — Filter rows first and groups afterwards

Let's filter rows and groups. Where keeps individual rows priced at least five before grouping. Having then keeps cities whose remaining sales reach one hundred. Florence and London qualify. Both thresholds are parameters.

In [ ]:
######## Filter rows and groups ########

selected = rows.query(
    columns="$city, ROUND(SUM($calculated_total), 2) AS sales",
    where="$unit_price >= :min_price", min_price=5,
    group_by="$city",
    having="SUM($calculated_total) >= :min_sales", min_sales=100,
    order_by="$city",
)
selected.fetch()

## 4 — Fetch rows and read their fields

Let's inspect a fetched row. Fetch returns a list of named rows. Read a field by its name, or turn a row into an ordinary Python dictionary with dict. Each fetch executes the query again.

In [ ]:
######## Inspect a fetched row ########

records = report.fetch()
records[0]
records[0]["sales"]
dict(records[0])

## 5 — Index the summaries by city

Let's look up a city directly. Fetch as dict indexes the result rows by the selected key. Our grouped cities are unique, so none are lost. The values are still named rows. With a repeated key, a later row would replace the earlier one.

In [ ]:
######## Look up a city directly ########

by_city = report.fetchAsDict(key="city")
by_city["Florence"]
by_city["Florence"]["sales"]

## 6 — Group detail rows in Python

Let's keep the detail rows too. Fetch grouped creates Python lists, one for each city. It does not calculate SQL aggregates. Florence keeps all twelve detail rows. Use this when you want groups of records rather than summary totals.

In [ ]:
######## Keep the detail rows too ########

details = rows.query(
    columns="$id, $city, $quantity, $calculated_total",
    order_by="$id",
)
groups = details.fetchGrouped(key="city")
[(city, len(items)) for city, items in groups.items()]
groups["Florence"][0]

## 7 — Fetch one record or just the keys

Let's fetch one known product. Fetch record returns a single row. This filter uses a unique key. A missing row raises an error unless ignoreMissing allows None. Multiple rows raise an error unless explicit order by selects the first ordered row.

In [ ]:
######## Fetch one known product ########

products.query(
    columns="$code, $name, $stock",
    where="$code = :code", code="TEA",
).fetchRecord()

Let's fetch just the product keys. Fetch pkeys returns a list of primary-key values in query order. Our limit selects the first three drink codes.

In [ ]:
######## Fetch just the product keys ########

products.query(
    where="$category = :category", category="Drinks",
    order_by="$code", limit=3,
).fetchPkeys()

## 8 — Get JSON text

Let's get the report as JSON. Fetch as JSON returns text. Decode it to inspect the first object. This creates no file. We will export JSON to a file shortly.

In [ ]:
######## Get the report as JSON ########

import json
payload = report.fetchAsJson()
type(payload).__name__
json.loads(payload)[0]

## 9 — Read larger results in batches

Let's fetch the details in batches. Iterfetch yields batches of rows. Here they contain ten, ten, ten and six records. We skip the final empty batch. This example records only the sizes; real code can process each batch as it arrives.

In [ ]:
######## Fetch the details in batches ########

[len(batch) for batch in details.iterfetch(arraysize=10) if batch]

## 10 — Export the report directly to CSV

Let's export the CSV report. Pass a destination to fetch. The file extension chooses the format, and the return value is seven: the number of rows written. The headings use our optional aliases. Only the report file is saved; SQLite is still in memory.

In [ ]:
######## Export the CSV report ########

report.fetch("sales-by-city.csv")
from pathlib import Path
print(Path("sales-by-city.csv").read_text())

## 11 — Export the same report to JSON

Let's export JSON as well. The same query writes a JSON array when the destination ends in dot json. Again, fetch returns a row count. Existing destination files are replaced. The notebook includes a way to download both reports.

In [ ]:
######## Export JSON as well ########

report.fetch("sales-by-city.json")
json.loads(Path("sales-by-city.json").read_text())[0]
db.commit()

## 12 — Thank you — from queries to reports

Thanks for following along! We grouped sales, filtered summaries, chose result formats and exported two reports. You can now try the optional best-buyer exercise, which reuses grouping to build a calculated relation, or go straight to lesson eight for writes and Python triggers.

```text
Thank you!
GROUP BY + HAVING → summaries
fetch / fetchAsDict / fetchGrouped → Python results
fetchRecord / fetchPkeys / fetchAsJson / iterfetch
fetch(destination) → CSV and JSON reports

Optional next: best buyer and a calculated relation
Or skip to lesson 8: writes and Python triggers
```

## Download your reports

The exported files are in the notebook's temporary folder. In Colab, run the cell below to download them before ending the session. In local Jupyter, it prints their full paths so you can open or copy them.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print(Path("sales-by-city.csv").resolve())
    print(Path("sales-by-city.json").resolve())
else:
    files.download("sales-by-city.csv")
    files.download("sales-by-city.json")


## Check your result

Run this final cell to check the exercise.

In [ ]:
from math import isclose
import csv
import json
from pathlib import Path
expected = {'Amsterdam': {'sales': 26.2, 'row_count': 3, 'ids': [10, 11, 12]}, 'Boston': {'sales': 57.8, 'row_count': 3, 'ids': [22, 23, 24]}, 'Florence': {'sales': 300.65, 'row_count': 12, 'ids': [1, 2, 3, 13, 14, 15, 25, 26, 27, 31, 32, 33]}, 'Helsinki': {'sales': 57.2, 'row_count': 6, 'ids': [7, 8, 9, 34, 35, 36]}, 'London': {'sales': 189.7, 'row_count': 6, 'ids': [4, 5, 6, 28, 29, 30]}, 'Manchester': {'sales': 87.4, 'row_count': 3, 'ids': [16, 17, 18]}, 'New York': {'sales': 96.3, 'row_count': 3, 'ids': [19, 20, 21]}}
records = report.fetch()
assert [r["city"] for r in records] == sorted(expected)
assert all(isclose(float(r["sales"]), expected[r["city"]]["sales"]) and r["row_count"] == expected[r["city"]]["row_count"] for r in records)
assert all(dict(report.fetchAsDict(key="city")[r["city"]]) == dict(r) for r in records)
assert json.loads(report.fetchAsJson()) == [dict(r) for r in records]
assert {city: [r["id"] for r in items] for city, items in details.fetchGrouped(key="city").items()} == {city: item["ids"] for city, item in expected.items()}
csv_rows = list(csv.DictReader(Path("sales-by-city.csv").open()))
assert len(csv_rows) == 7
assert all(r["city"] == src["city"] and isclose(float(r["sales"]), src["sales"]) and int(r["row_count"]) == src["row_count"] for r, src in zip(csv_rows, records))
assert json.loads(Path("sales-by-city.json").read_text()) == [dict(r) for r in records]
assert db.table("shop.product").query().count() == 30
assert db.table("shop.customer").query().count() == 8
assert db.table("shop.invoice").query().count() == 12
assert rows.query().count() == 36
db.commit()

db.commit()
db.commit()
print("Lesson checks passed.")
